# Frequency Domain: Trapping Square

In [1]:
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as pl
import numpy as np

In [14]:
# domain parameters
Rout = 2
RScat = 1
DScat = 0.1
DHole = 0.2
R=1.5

#Method default parameters
maxh = 0.1  
order = 4
alpha = 1j #pml parameter

#Problem parameters
k = 10.8        #10.8 resonant, 10.6 non-resonant
theta = 0

In [3]:
def create_geo(Rout = Rout, RScat = RScat, DScat = DScat, DHole = DHole, R = R, draw = True):
    domain = Circle((0, 0), Rout).Face()
    domain.edges.name = 'outer'
    domain.faces.name = 'outer'
    
    inner = Circle((0, 0), R).Face()
    inner.edges.name = 'interface'
    inner.faces.name = 'inner'

    scatterer = (
            MoveTo(-RScat/2,-RScat/2).Rectangle(RScat,RScat).Face()
            -MoveTo(-RScat/2+DScat/2,-RScat/2+DScat/2).Rectangle(RScat-DScat,RScat-DScat).Face()
            -MoveTo(-RScat/2-DScat,-DHole/2).Rectangle(2*DScat,DHole).Face()
        )
    scatterer.edges.name = 'scatterer'

    scattererdom = MoveTo(-RScat/2,-RScat/2).Rectangle(RScat,RScat).Face()-scatterer
    scattererdom.faces.name = 'scatterer'
    


    if draw:
        Draw(Glue([domain-inner,inner-scattererdom-scatterer, scattererdom-scatterer]))
    geo = OCCGeometry(Glue([domain-inner,inner-scattererdom-scatterer, scattererdom]), dim=2)
    return geo

In [4]:
geo = create_geo(draw = False)
mesh = Mesh(geo.GenerateMesh(maxh=maxh))
mesh.Curve(order)
Draw(mesh)
print(mesh.GetMaterials(),mesh.GetBoundaries())

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

('outer', 'inner', 'scatterer') ('outer', 'interface', 'default', 'scatterer', 'scatterer', 'scatterer', 'scatterer', 'scatterer', 'scatterer', 'scatterer', 'scatterer', 'scatterer', 'scatterer', 'scatterer', 'scatterer')


In [15]:
def solve_scattering(maxh = maxh, order = order, k = k, theta = theta, draw = True):
    geo = create_geo(Rout = Rout, RScat = RScat, DScat = DScat, DHole = DHole, R = R, draw = False)
    mesh = Mesh(geo.GenerateMesh(maxh=maxh))
    mesh.Curve(order)

    fes = H1(mesh,order = order,complex = True,dirichlet = mesh.Boundaries("scatterer"))

    w,w_ = fes.TnT()

    rho = sqrt(x**2+y**2)

    #Omega = CF([(Rout-rho)/(Rout-R),1,1])
    Omega = mesh.MaterialCF({"outer": (Rout-rho)/(Rout-R)},default=1)

    DOmega = Omega.Diff(rho)
    L = Omega-rho*DOmega
    G = Omega**2/L

    #H = CF([1 - G,0,0]) #characteristic preserving
    H = mesh.MaterialCF({"outer": 1-G},default=0) #characteristic preserving
    M = 1+H  #(1+H^2)/G, simplifies to 1+H for characteristic preserving


    vx = CF((x,y))

    Pi = OuterProduct(vx,vx)/rho**2 #projection onto radial vector
    Pi_perp = Id(2)-Pi              #projection onto tangential vector

    ### k**2
    M_ = M*w*w_*dx

    ### 1j*k
    C_ = (
        -H/rho*w*InnerProduct(vx,grad(w_))*dx
        +H/rho*w_*InnerProduct(vx,grad(w))*dx
        +w_*w*ds('outer')
        )

    ## 1
    K_ = (
        -grad(w)*( (G*Pi+L*Pi_perp)*grad(w_))*dx
        -Omega/L/2/rho*DOmega*InnerProduct(grad(w),vx)*w_*dx
        -Omega/L/2/rho*DOmega*InnerProduct(grad(w_),vx)*w*dx
        -1/L/4*DOmega**2*w*w_*dx
        )

    A = BilinearForm(k**2*M_+1j*k*C_+K_).Assemble().mat
    Ainv = A.Inverse(freedofs=fes.FreeDofs())

    direction = CF((cos(theta),sin(theta)))

    g = exp(1j*k*InnerProduct(direction,(x,y))/Norm(direction))
    gfu = GridFunction(fes)

    gfu.Set(g, definedon=mesh.Boundaries('scatterer'))
    res = - A * gfu.vec
    gfu.vec.data += Ainv*res
    if draw:
        #Draw(gfu,mesh, "scattered field",min=-1, max=1)
        print('total field hyperboloidal')
        Draw(g-gfu,mesh,"total field", min=-1, max=1)
        #Draw(g,mesh,"incoming field")
    gfutotal = GridFunction(fes)
    gfutotal.Set(g)
    gfutotal.vec.data -= gfu.vec
    return gfu,gfutotal

In [16]:
def solve_scattering_pml(maxh = maxh, order = order, k = k, theta = theta, draw = True, alpha = alpha):
    geo = create_geo(Rout = Rout, RScat = RScat, DScat = DScat, DHole = DHole, R = R, draw = False)
    mesh = Mesh(geo.GenerateMesh(maxh=maxh))
    mesh.Curve(order+1)
    
    mesh.SetPML(pml.Radial((0,0),R,alpha),'outer')

    fes = H1(mesh,order = order,complex = True,dirichlet = mesh.Boundaries("scatterer"))

    w,w_ = fes.TnT()
    M_ = w*w_*dx(bonus_intorder=4)    #bonus_intorder necessary due to additional coefficients from trafos (hidden in PML mesh trafo)

    K_ = (
        -grad(w)*grad(w_)*dx(bonus_intorder=4)
        )

    A = BilinearForm(k**2*M_+K_).Assemble().mat
    
    Ainv = A.Inverse(freedofs=fes.FreeDofs())

    direction = CF((cos(theta),sin(theta)))

    g = exp(1j*k*InnerProduct(direction,(x,y))/Norm(direction))
    gfu = GridFunction(fes)

    gfu.Set(g, definedon=mesh.Boundaries('scatterer'))
    res = - A * gfu.vec
    gfu.vec.data += Ainv*res
    if draw:
        #Draw(gfu,mesh, "scattered field",min=-1, max=1)
        print('total field PML')
        Draw(g-gfu,mesh,"total field", min=-1, max=1)
        #Draw(g,mesh,"incoming field")
    gfutotal = GridFunction(fes)
    gfutotal.Set(g)
    gfutotal.vec.data -= gfu.vec
    return gfu,gfutotal

In [19]:
gfu,gfu_total = solve_scattering(k=10.8,theta = pi/4,draw=True)  #resonant
gfu,gfu_total = solve_scattering(k=10.6,theta = pi/4,draw=True)  #non-resonant

total field hyperboloidal


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'sp…

total field hyperboloidal


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'sp…

In [20]:
gfu_pml,gfu_total_pml = solve_scattering_pml(k=10.8,theta = pi/4,draw=True)
gfu_pml,gfu_total_pml = solve_scattering_pml(k=10.6,theta = pi/4,draw=True)

total field PML


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'sp…

total field PML


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'sp…